# `@tool` y `StructuredTool.from_function`

En `herramientas_personalizadas.py` aparecen dos formas de convertir una función de Python en una herramienta de LangChain:

1. El decorador **`@tool`**.
2. **`StructuredTool.from_function`**.

Las dos fabrican el mismo tipo de objeto: un `StructuredTool`. El decorador es un atajo. Por dentro llama a `from_function`. Por eso en el curso se dice que la segunda forma es redundante y casi no se usa.

## Qué ve el modelo

Una herramienta no es solo la función. El modelo necesita tres datos para decidir si la llama y con qué argumentos:

- **name**: el nombre con el que la invoca.
- **description**: cuándo usarla. Si no pasas una, LangChain toma el docstring.
- **esquema de argumentos**: los campos que debe rellenar. Aquí hay uno, `query: str`.

Por eso se llama *estructurada*: el modelo no manda un texto suelto. Manda un JSON, por ejemplo `{"query": "Consulta de prueba"}`.

## 1. Con el decorador `@tool`

Esta es la forma del archivo. El decorador sustituye la función: a partir de esa línea, `herramienta_personalizada` es la herramienta, no una función normal. Se ejecuta con `.run()` o `.invoke()`.

In [ ]:
from langchain_core.tools import tool

@tool("Herramienta acceso base de datos usuarios", return_direct=True)
def herramienta_personalizada(query: str) -> str:
    """Consulta la base de datos de usuarios de la empresa."""
    return f"Respuesta a la consulta: {query}"

print(herramienta_personalizada.run("Consulta de prueba"))
print("Nombre:", herramienta_personalizada.name)
print("Descripcion:", herramienta_personalizada.description)
print("Tipo:", type(herramienta_personalizada).__name__)

Dos detalles de esta versión:

- El nombre que ve el modelo es el string del decorador (`Herramienta acceso base de datos usuarios`), no el nombre de la función en Python.
- `return_direct=True` solo importa dentro de un agente. Si el agente llama a esta herramienta, su texto vuelve al usuario tal cual, sin una segunda pasada del modelo. En un `.run()` suelto no cambia nada.

## 2. Con `StructuredTool.from_function`

Aquí la función se queda como función. `from_function` construye la herramienta al lado y la guardas en otra variable.

In [ ]:
from langchain_core.tools import StructuredTool

def herramienta_personalizada_estructurada(query: str) -> str:
    """Consulta la base de datos de usuarios de la empresa."""
    return f"Respuesta a la consulta: {query}"

mi_tool = StructuredTool.from_function(herramienta_personalizada_estructurada)

print(mi_tool.run("Consulta de prueba"))
print("Nombre:", mi_tool.name)
print("Descripcion:", mi_tool.description)
print("Tipo:", type(mi_tool).__name__)
print("La funcion original sigue siendo funcion:", callable(herramienta_personalizada_estructurada))

Si no pasas `name`, el nombre de la herramienta es el de la función. `return_direct` queda en `False`, que es el valor por defecto: un agente recibiría el resultado y el modelo podría seguir hablando.

## 3. Son el mismo objeto

Las dos celdas imprimen `StructuredTool`. El decorador, con la configuración por defecto (`infer_schema=True`), hace esto por dentro:

`StructuredTool.from_function(func, name=..., description=..., return_direct=...)`

El esquema sale igual en las dos: LangChain lee `query: str` y el docstring, y arma un modelo de Pydantic. Por eso el curso llama redundante a escribir `from_function` a mano.

In [ ]:
print("@tool")
print("  tipo:          ", type(herramienta_personalizada).__name__)
print("  name:          ", herramienta_personalizada.name)
print("  description:   ", herramienta_personalizada.description)
print("  return_direct: ", herramienta_personalizada.return_direct)
print("  args:          ", herramienta_personalizada.args)

print()
print("from_function")
print("  tipo:          ", type(mi_tool).__name__)
print("  name:          ", mi_tool.name)
print("  description:   ", mi_tool.description)
print("  return_direct: ", mi_tool.return_direct)
print("  args:          ", mi_tool.args)

Lo que cambia en estos dos snippets no es el mecanismo. Cambia lo que se configuró:

| | `@tool` del archivo | `from_function` sin argumentos extra |
|---|---|---|
| Tipo | `StructuredTool` | `StructuredTool` |
| Nombre | El string del decorador | El nombre de la función |
| Descripción | El docstring | El docstring |
| Argumentos | `query: str` | `query: str` |
| `return_direct` | `True` | `False` |
| La función de Python | Queda reemplazada por la tool | Sigue existiendo aparte |

`.run("Consulta de prueba")` funciona en las dos porque hay un solo argumento de tipo `str`. La forma explícita, la que coincide con el JSON que arma el modelo, es `.invoke({"query": "Consulta de prueba"})`.

## 4. Dejarlas iguales

`from_function` acepta los mismos datos que el decorador. Con `name` y `return_direct` las dos herramientas quedan configuradas igual. Sigue siendo más código para el mismo resultado, que es la razón de preferir `@tool`.

In [ ]:
tool_explicita = StructuredTool.from_function(
    func=herramienta_personalizada_estructurada,
    name="Herramienta acceso base de datos usuarios",
    return_direct=True,
)

print(tool_explicita.name)
print(tool_explicita.return_direct)
print(tool_explicita.invoke({"query": "Consulta de prueba"}))

## 5. Cuándo sí usar `from_function`

Casi nunca en código de un curso o de una app propia. Tiene sentido en tres casos:

- La función vive en otro módulo y no puedes ponerle el decorador.
- Quieres conservar la función normal y, además, la herramienta.
- Construyes varias herramientas en un bucle, a partir de una lista de funciones.

Para una función que escribes tú, `@tool` es la forma que se usa.

## 6. No es lo mismo que `Tool` de `herramientas.py`

`Tool` (el de `PythonREPL`) es la herramienta simple: entra un string y sale un string. No publica un campo `query`.

`@tool` y `StructuredTool.from_function` publican el esquema. El modelo rellena `query` como un argumento con nombre. Por eso las dos de este notebook sirven igual en `bind_tools`, como en `herramientas_personalizadas_llm.py`.

## 7. El campo `args_schema` (Pydantic)

El curso lo menciona y lo deja pasar. El campo se llama `args_schema`. Sirve para entregar tú el modelo de Pydantic con los argumentos de la herramienta, en vez de dejar que LangChain lo deduzca de la función.

En este ejemplo no cambia el comportamiento, y por eso no hace falta verlo para seguir:

- La función ya dice `query: str`.
- Con `infer_schema=True` (el valor por defecto de `@tool` y de `from_function`), LangChain construye un `BaseModel` a partir de esa anotación.
- Ese modelo es el que ya viste en `.args`: un campo `query` de tipo string.
- Escribir la clase a mano repetiría el mismo campo. `.run()` y `.invoke()` siguen igual.

Pydantic aquí no describe la respuesta del modelo, como en el `PydanticOutputParser` del Tema 2. Describe lo que el modelo tiene que rellenar para llamar a la herramienta.

Tiene sentido cuando la firma de la función se queda corta:

- Quieres una descripción por argumento (`Field(description=...)`). El docstring describe la herramienta entera, no cada campo.
- Hay validación: un entero entre 0 y 100, un `Literal`, un campo opcional.
- Hay varios argumentos, o un objeto anidado, y el modelo necesita saber cómo armarlos.
- Al heredar de `BaseTool`, `args_schema` es la forma normal de declarar las entradas, porque no hay una función de la que inferirlas.


In [ ]:
from pydantic import BaseModel, Field

class ConsultaUsuarios(BaseModel):
    query: str = Field(
        description="Texto de la consulta sobre un usuario, por ejemplo un id como UX341234."
    )

@tool(
    "Herramienta acceso base de datos usuarios",
    args_schema=ConsultaUsuarios,
    return_direct=True,
)
def herramienta_con_schema(query: str) -> str:
    """Consulta la base de datos de usuarios de la empresa."""
    return f"Respuesta a la consulta: {query}"

print("Inferido de query: str")
print(herramienta_personalizada.args)
print()
print("Escrito con args_schema")
print(herramienta_con_schema.args)
print()
print(herramienta_con_schema.invoke({"query": "Consulta de prueba"}))
